# I-N110 → I-N120 멀티모달 nnU-Net 연속 실험

팀원의 I-N00/I-N10과 기존 I-N20/I-N30/I-N40 결과는 수정하지 않는다. 이 노트북은 **I-N110 Dual-Stem** 완료 후 **I-N120 Dual-Encoder + multi-scale gated fusion**을 자동으로 이어서 실행한다.

공통 조건: DWI+ADC, fold 0, 300 epochs, early stopping 없음, epoch 150부터 10 epoch 간격 full-volume validation, Top-3 anchor의 ±5 epoch 재평가, best 확률맵 저장, threshold sweep.

In [ ]:
from pathlib import Path
import json, os, signal, subprocess, time
import pandas as pd
from IPython.display import display

REPO = Path("/home/stu03/workspace/model")
PYTHON = REPO / ".venv/bin/python"
assert PYTHON.is_file(), PYTHON

EXPERIMENTS = ["I-N110", "I-N120"]
CONFIGS = {x: REPO / f"configs/experiments/ischemia/{x}.yaml" for x in EXPERIMENTS}
CONFIGS

## 모델 차이

- **I-N110**: DWI/ADC별 초기 stem을 학습한 뒤 feature를 합쳐 shared residual encoder/decoder로 전달한다.
- **I-N120**: DWI/ADC 전체 encoder를 독립 학습하고 각 scale에서 learnable gate로 feature를 합친 뒤 shared decoder로 전달한다.
- 비교 기준과 split은 기존 I-N20~I-N40과 동일하다.

In [ ]:
# 실제 학습 없이 모든 경로와 명령 확인
subprocess.run([str(PYTHON), str(REPO / "scripts/ischemia/training/run_i_n_multimodal_sequence.py"), "--dry-run"], cwd=REPO, check=True)

## 연속 학습 시작/재개

아래 셀을 한 번 실행하면 background에서 I-N110을 수행하고 정상 완료된 경우에만 I-N120으로 넘어간다. checkpoint가 있으면 자동 재개하며, 이미 최종 평가와 확률맵까지 끝난 실험은 건너뛴다.

In [ ]:
LOG = REPO / "runs/I-nnunet/I-N110_N120_sequence.log"
PID_FILE = REPO / "runs/I-nnunet/I-N110_N120_sequence.pid"
PID_FILE.parent.mkdir(parents=True, exist_ok=True)
if PID_FILE.is_file():
    old_pid = int(PID_FILE.read_text())
    try:
        os.kill(old_pid, 0); raise RuntimeError(f"이미 실행 중입니다: PID={old_pid}")
    except ProcessLookupError: pass
handle = LOG.open("a", buffering=1)
proc = subprocess.Popen([str(PYTHON), str(REPO / "scripts/ischemia/training/run_i_n_multimodal_sequence.py")], cwd=REPO, stdout=handle, stderr=subprocess.STDOUT, start_new_session=True)
PID_FILE.write_text(str(proc.pid))
print(f"started PID={proc.pid} | log={LOG}")

## 실시간 진행률

In [ ]:
import yaml
from IPython.display import clear_output

def process_alive(pid_file):
    if not pid_file.is_file(): return False
    try:
        os.kill(int(pid_file.read_text()), 0); return True
    except (ProcessLookupError, ValueError): return False

try:
    while True:
        rows=[]
        for eid in EXPERIMENTS:
            c=yaml.safe_load(CONFIGS[eid].read_text()); e,p=c["experiment"],c["plans"]
            fold=Path(c["paths"]["results_root"])/f"Dataset{int(e['dataset_id']):03d}_{e['dataset_name']}"/f"nnUNetTrainerIschemiaMultimodal__{p['output_identifier']}__{e['configuration']}"/f"fold_{e['fold']}"
            progress=fold/"progress.json"; row={"experiment":eid,"result":str(fold),"status":"waiting"}
            if progress.is_file(): row.update(json.loads(progress.read_text()))
            row["probability_maps"]=len(list((fold/"best_probability_validation").glob("*.npz")))
            rows.append(row)
        clear_output(wait=True)
        print(time.strftime("Updated: %Y-%m-%d %H:%M:%S"))
        display(pd.DataFrame(rows).reindex(columns=["experiment","status","current_epoch","total_epochs","progress_percent","eta_seconds","best_full_macro_dice","best_full_macro_epoch","probability_maps","result"]))
        if LOG.is_file(): print("\n--- latest log ---\n"+"".join(LOG.read_text(errors="replace").splitlines(True)[-20:]))
        if not process_alive(PID_FILE):
            print("\n연속 실행 프로세스가 종료되었습니다. 마지막 상태입니다."); break
        time.sleep(10)
except KeyboardInterrupt:
    print("모니터만 중단했습니다. background 학습은 계속됩니다.")

## 중단

필요할 때만 실행한다. sequence와 현재 child 학습 프로세스 그룹에 SIGTERM을 보내며 checkpoint는 보존된다.

In [ ]:
pid=int(PID_FILE.read_text()); os.killpg(pid, signal.SIGTERM); print(f"stop requested: {pid}")

## 완료 결과

각 실험 폴더의 `final_metrics`, `final_selection.json`, `best_probability_validation`, `threshold_summary.json`을 확인한다.

In [ ]:
result_rows=[]
for row in rows:
    fold=Path(row["result"]); sel=fold/"final_selection.json"; th=fold/"final_metrics/threshold_summary.json"
    out={"experiment":row["experiment"],"completed":sel.is_file(),"probability_maps":len(list((fold/"best_probability_validation").glob("*.npz")))}
    if sel.is_file():
        x=json.loads(sel.read_text())["selected"]; out.update(selected_epoch=x.get("epoch"),macro_dice=x.get("macro_dice_positive"),lesion_recall=x.get("lesion_detection_rate"))
    if th.is_file(): out["recommended_threshold"]=json.loads(th.read_text())["recommended_lesion_f2"]["threshold"]
    result_rows.append(out)
display(pd.DataFrame(result_rows))